# Page Expansion — Is It Worth It?

**Question**: If we expand each retrieved chunk to include all sibling chunks from the same `(doc_name, page_num)`, do we actually pick up content that helps answer queries — or does it just add noise?

**Approach**: Use the live Qdrant store. Three analyses:

1. **Chunk fragmentation** — how often does a single page produce multiple chunks, and how small are the tiny ones (likely headings / icon-only chunks)?
2. **Top-fragmented pages** — show what the chunks on the most-split pages actually contain, to judge whether sibling chunks carry meaning.
3. **Retrieval simulation** — run real queries; show what `retrieve_pages` returns vs. what page expansion would add. Includes the Mission 3 example.

Decision criterion: if a meaningful fraction of retrieved top-k chunks have unretrieved siblings that contain content a human would want when answering the question, page expansion is worth implementing.

## Setup

In [2]:
import sys
from collections import defaultdict, Counter
from pathlib import Path

# Make the package importable from the notebook
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qdrant_client import models

from boardgame_agent.config import COLLECTION_NAME
from boardgame_agent.rag.indexer import get_qdrant_client
from boardgame_agent.rag.retriever import retrieve_pages, format_pages_for_llm

client = get_qdrant_client()
print(f"Collection: {COLLECTION_NAME}")
print(f"Exists: {client.collection_exists(COLLECTION_NAME)}")

/Users/micahshanks/Dev/board-game-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection: rulebook_pages
Exists: True


In [3]:
# ── Choose a game to analyze ──────────────────────────────
GAME_ID = "the_crew__the_quest_for_planet_nine"
# ──────────────────────────────────────────────────────────

# Show available game_ids in the collection so it's easy to switch.
scroll_iter = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=10_000,
    with_payload=["game_id"],
    with_vectors=False,
)
all_game_ids = Counter(p.payload["game_id"] for p in scroll_iter[0])
print("game_id -> chunk count")
for gid, n in sorted(all_game_ids.items(), key=lambda x: -x[1]):
    marker = " ← selected" if gid == GAME_ID else ""
    print(f"  {gid:50s} {n}{marker}")

game_id -> chunk count
  dungeons___dragons                                 5420
  gloomhaven                                         198
  sky_team                                           168
  the_crew__the_quest_for_planet_nine                68 ← selected
  the_lord_of_the_rings__duel_for_middle_earth       44


In [ ]:
# Fetch every chunk for the selected game (full payloads, no vectors)
def fetch_all_chunks_for_game(game_id: str) -> list[dict]:
    chunks = []
    offset = None
    while True:
        points, offset = client.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=models.Filter(
                must=[models.FieldCondition(key="game_id", match=models.MatchValue(value=game_id))]
            ),
            limit=512,
            offset=offset,
            with_payload=True,
            with_vectors=False,
        )
        chunks.extend({"id": p.id, **p.payload} for p in points)
        if offset is None:
            break
    return chunks

chunks = fetch_all_chunks_for_game(GAME_ID)
print(f"Total chunks for {GAME_ID}: {len(chunks)}")
print(f"Documents: {sorted(set(c['doc_name'] for c in chunks))}")
print(f"Doc tags:  {sorted(set(c.get('doc_tag', '(none)') for c in chunks))}")

## 1. Chunk fragmentation per page

How many chunks does each page produce? A page that produces 1 chunk needs no expansion. A page that produces 3+ chunks is where expansion might help.

In [ ]:
# Group chunks by (doc_name, page_num)
page_groups: dict[tuple[str, int], list[dict]] = defaultdict(list)
for c in chunks:
    page_groups[(c["doc_name"], c["page_num"])].append(c)

chunks_per_page = Counter(len(v) for v in page_groups.values())

print(f"Total unique pages: {len(page_groups)}")
print()
print("Distribution of chunks-per-page:")
print(f"{'chunks':>6s}  {'pages':>6s}  {'% pages':>8s}")
total_pages = sum(chunks_per_page.values())
for n in sorted(chunks_per_page):
    pages = chunks_per_page[n]
    pct = 100 * pages / total_pages
    bar = "█" * int(pct / 2)
    print(f"{n:>6d}  {pages:>6d}  {pct:>6.1f}%  {bar}")

multi_chunk_pages = sum(v for k, v in chunks_per_page.items() if k > 1)
print(f"\nPages with >1 chunk: {multi_chunk_pages} / {total_pages} ({100*multi_chunk_pages/total_pages:.1f}%)")
print("→ Page expansion only matters for these pages. If this number is low, the change is low-value.")

In [ ]:
# How small are individual chunks? Tiny ones (single-bbox or very short text)
# are the strongest case for expansion — they're the ones most likely to be
# a stranded heading, caption, or icon row.
import statistics

chunk_text_lens = [len(c.get("text", "")) for c in chunks]
bbox_counts = [len(c.get("bboxes", [])) for c in chunks]
picture_bbox_counts = [
    sum(1 for b in c.get("bboxes", []) if b.get("label") == "picture") for c in chunks
]

print("Chunk text length (chars):")
print(f"  min/median/mean/max: {min(chunk_text_lens)} / {int(statistics.median(chunk_text_lens))} / {int(statistics.mean(chunk_text_lens))} / {max(chunk_text_lens)}")
print()
print("Bboxes per chunk:")
print(f"  min/median/mean/max: {min(bbox_counts)} / {int(statistics.median(bbox_counts))} / {statistics.mean(bbox_counts):.1f} / {max(bbox_counts)}")
print()

tiny = [c for c in chunks if len(c.get("text", "")) < 100]
single_bbox = [c for c in chunks if len(c.get("bboxes", [])) <= 1]
picture_only = [
    c for c in chunks
    if c.get("bboxes") and all(b.get("label") == "picture" for b in c["bboxes"])
]

print(f"Tiny chunks (<100 chars):       {len(tiny):>4d} / {len(chunks)} ({100*len(tiny)/len(chunks):.1f}%)")
print(f"Single-bbox chunks:             {len(single_bbox):>4d} / {len(chunks)} ({100*len(single_bbox)/len(chunks):.1f}%)")
print(f"Picture-only chunks:            {len(picture_only):>4d} / {len(chunks)} ({100*len(picture_only)/len(chunks):.1f}%)")
print()
print("→ Tiny / single-bbox / picture-only chunks are the strongest case for expansion:")
print("   on their own they retrieve a fragment; with siblings attached they retrieve")
print("   the surrounding section.")

## 2. Top-fragmented pages — what's actually on them?

Show the pages with the most chunks. Scan the chunk text to judge: do these chunks belong together (expansion helps), or are they genuinely independent sub-sections (expansion adds noise)?

In [ ]:
TOP_N_PAGES = 8
TEXT_PREVIEW_CHARS = 180

most_fragmented = sorted(page_groups.items(), key=lambda kv: -len(kv[1]))[:TOP_N_PAGES]

for (doc_name, page_num), page_chunks in most_fragmented:
    print("=" * 80)
    print(f"DOC: {doc_name}  PAGE: {page_num}  CHUNKS: {len(page_chunks)}")
    print("=" * 80)
    for i, c in enumerate(page_chunks):
        text = c.get("text", "")
        preview = text[:TEXT_PREVIEW_CHARS].replace("\n", " ")
        n_bbox = len(c.get("bboxes", []))
        n_pic = sum(1 for b in c.get("bboxes", []) if b.get("label") == "picture")
        labels = Counter(b.get("label", "?") for b in c.get("bboxes", []))
        label_str = ", ".join(f"{k}={v}" for k, v in labels.items())
        print(f"\n  [{i}] {len(text):>4d} chars  |  bboxes: {n_bbox} ({label_str})  |  pics: {n_pic}")
        print(f"      {preview!r}")
    print()

## 3. Retrieval simulation — what does expansion add for real queries?

Run the existing `retrieve_pages` for a few sample queries. For each top-k hit, fetch its sibling chunks (same `doc_name + page_num`) that weren't already returned. Show the gained content so you can judge whether it's meaningful or noise.

In [8]:
def fetch_siblings(doc_name: str, page_num: int, exclude_ids: set) -> list[dict]:
    """Return all chunks on (doc_name, page_num) NOT already in exclude_ids."""
    points, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=models.Filter(
            must=[
                models.FieldCondition(key="game_id", match=models.MatchValue(value=GAME_ID)),
                models.FieldCondition(key="doc_name", match=models.MatchValue(value=doc_name)),
                models.FieldCondition(key="page_num", match=models.MatchValue(value=page_num)),
            ]
        ),
        limit=64,
        with_payload=True,
        with_vectors=False,
    )
    return [{"id": p.id, **p.payload} for p in points if p.id not in exclude_ids]


def show_retrieval_with_expansion(query: str, doc_tag: str | None = None, k: int = 5):
    print("#" * 80)
    print(f"QUERY: {query!r}    (doc_tag={doc_tag!r}, k={k})")
    print("#" * 80)
    points = retrieve_pages(client, query, GAME_ID, k=k, doc_tag=doc_tag)
    retrieved_ids = {p.id for p in points}

    total_sibling_chars = 0
    total_sibling_chunks = 0
    total_sibling_pictures = 0

    for rank, p in enumerate(points, 1):
        pl = p.payload
        doc_name = pl["doc_name"]
        page_num = pl["page_num"]
        text = pl.get("text", "")
        print(f"\n── #{rank}  {doc_name} p.{page_num}  ({len(text)} chars)  doc_tag={pl.get('doc_tag')}")
        print(f"   RETRIEVED CHUNK: {text[:240]!r}")

        siblings = fetch_siblings(doc_name, page_num, retrieved_ids)
        if not siblings:
            print("   (no siblings on this page — expansion would add nothing)")
            continue

        for j, s in enumerate(siblings):
            stext = s.get("text", "")
            n_pic = sum(1 for b in s.get("bboxes", []) if b.get("label") == "picture")
            total_sibling_chars += len(stext)
            total_sibling_chunks += 1
            total_sibling_pictures += n_pic
            print(f"   + SIBLING [{j}] {len(stext)} chars, {n_pic} pictures: {stext[:240]!r}")

    print("\n" + "-" * 80)
    print(f"EXPANSION ADDS: {total_sibling_chunks} chunks, {total_sibling_chars} chars, {total_sibling_pictures} picture bboxes")
    print("-" * 80)

In [ ]:
# Mission 3 — the canonical cross-reference failure case
show_retrieval_with_expansion(
    "Can you explain the criteria for winning on mission 3",
    doc_tag=None,
    k=5,
)

In [ ]:
# Text-only baseline — sibling adds should be modest or zero here
show_retrieval_with_expansion(
    "who starts the game",
    doc_tag=None,
    k=5,
)

In [ ]:
# Icon-meaning question — does the retrieved chunk land you on a page where
# the icon definition is a sibling chunk that wouldn't otherwise be returned?
show_retrieval_with_expansion(
    "what does the red starburst symbol mean on mission cards",
    doc_tag=None,
    k=5,
)

In [ ]:
# Another multi-source mission question to test pattern consistency
show_retrieval_with_expansion(
    "how do you win mission 12",
    doc_tag=None,
    k=5,
)

## 4. Aggregate signal across many queries

Run a batch of queries and quantify: across all top-k results, what fraction have at least one sibling chunk that would be added by expansion? What's the average sibling-chars-added per retrieval?

**Edit `QUERIES` to test your own corpus.**

In [ ]:
QUERIES = [
    "who starts the game",
    "how do you win mission 3",
    "how do you win mission 12",
    "what is the commander",
    "what does the red starburst symbol mean",
    "how does communication work",
    "setup for 4 players",
    "what is a trick",
    "distress signal rules",
    "task tokens",
]

results = []
for q in QUERIES:
    points = retrieve_pages(client, q, GAME_ID, k=5, doc_tag=None)
    retrieved_ids = {p.id for p in points}
    for p in points:
        pl = p.payload
        siblings = fetch_siblings(pl["doc_name"], pl["page_num"], retrieved_ids)
        sibling_chars = sum(len(s.get("text", "")) for s in siblings)
        sibling_pictures = sum(
            sum(1 for b in s.get("bboxes", []) if b.get("label") == "picture")
            for s in siblings
        )
        results.append({
            "query": q,
            "doc": pl["doc_name"],
            "page": pl["page_num"],
            "chunk_chars": len(pl.get("text", "")),
            "n_siblings": len(siblings),
            "sibling_chars": sibling_chars,
            "sibling_pictures": sibling_pictures,
        })

n_with_siblings = sum(1 for r in results if r["n_siblings"] > 0)
n_with_picture_siblings = sum(1 for r in results if r["sibling_pictures"] > 0)

print(f"Total (query, top-k-hit) pairs:                 {len(results)}")
print(f"  with ≥1 sibling chunk:                        {n_with_siblings} ({100*n_with_siblings/len(results):.0f}%)")
print(f"  with ≥1 sibling picture-bbox:                 {n_with_picture_siblings} ({100*n_with_picture_siblings/len(results):.0f}%)")
print()
avg_sibling_chars = sum(r["sibling_chars"] for r in results) / len(results)
print(f"Avg sibling chars added per retrieved chunk:    {avg_sibling_chars:.0f}")
print()
print("Per-query breakdown:")
print(f"{'query':<50s}  {'hits w/siblings':>16s}  {'+chars':>8s}  {'+pics':>6s}")
by_query: dict[str, list[dict]] = defaultdict(list)
for r in results:
    by_query[r["query"]].append(r)
for q, rs in by_query.items():
    n = sum(1 for r in rs if r["n_siblings"] > 0)
    chars = sum(r["sibling_chars"] for r in rs)
    pics = sum(r["sibling_pictures"] for r in rs)
    print(f"{q[:48]:<50s}  {n:>5d}/{len(rs):<10d}  {chars:>8d}  {pics:>6d}")

## 5. Verdict

Look at the numbers from sections 1–4 and judge:

- **If §1 shows most pages have 1 chunk** → page expansion is a no-op for most queries. Probably not worth implementing.
- **If §2 shows fragmented pages contain related content split across chunks** (heading + body, flavor text + icon rows) → expansion is likely high-value.
- **If §2 shows fragmented pages contain genuinely independent sub-sections** (different rules, different topics) → expansion will add noise.
- **If §4 shows most retrieved chunks have siblings that include picture-bboxes** → expansion specifically helps the icon-cross-reference failure mode you described.
- **If §4 shows the Mission 3 query already retrieves all relevant chunks (no siblings to add)** → the failure mode is elsewhere (rerank, prompts, decomposition) and page expansion alone won't fix it.

The honest answer comes from the data, not the diagnosis. If this notebook shows expansion adds <10% sibling content on average and the Mission 3 case isn't materially improved, prioritize a different change first.

## 6. Experiment — semantic icon labels

**Hypothesis**: if picture bboxes carry semantic labels (e.g. `Mission 3` instead of `black pentagon with the number 3`), text retrieval will land on the correct logbook pages for mission queries — because bbox text flows into the embedded chunk text via [`extractor.py:329`](../boardgame_agent/rag/extractor.py#L329).

**MVP scope**: only one icon class — pentagons-with-numbers, regex-matched against existing VLM descriptions. This isolates the mechanism. The full proposal would also label red starbursts (number of Order cards, defined on logbook p.3), task-order squares (defined on rulebook p.14), etc. — each of which requires its own discovery step. We don't do that here; we test whether labels-as-mechanism works at all.

**What this experiment proves / doesn't prove**:
- ✅ Proves whether semantic bbox labels can move correct pages into top-k for mission queries.
- ❌ Does **not** prove the agent can correctly *answer* the question. Answering Mission 3 also requires labels for the red starburst (= number of tasks) and the task-order squares — which this MVP doesn't add.

**This is reversible** — `reindex_all()` rebuilds from cached Docling JSONs. The cached JSON files are not modified.

In [4]:
# Augment pentagon picture bboxes in memory with 'Mission N' labels.
import re
import json
import sqlite3
from pathlib import Path

from boardgame_agent.config import DATA_DIR

# Match VLM descriptions like 'pentagon with the number 3', 'pentagon-shaped icon with the number 16'.
PENTAGON_PATTERN = re.compile(r'pentagon\b.*?\bnumber\s+(\d+)', re.IGNORECASE)

# doc_tag is injected into page dicts at index time (sidebar.py:370); the cached
# JSON doesn't carry it, so fetch from SQLite and re-inject before chunking.
conn = sqlite3.connect(str(DATA_DIR / 'games.db'))
conn.row_factory = sqlite3.Row
doc_tags = {
    r['doc_name']: r['doc_tag']
    for r in conn.execute('SELECT doc_name, doc_tag FROM documents WHERE game_id = ?', (GAME_ID,))
}
conn.close()

logbook_doc_name = next(name for name in doc_tags if 'Log_Book' in name)
logbook_tag = doc_tags[logbook_doc_name]
logbook_json = DATA_DIR / 'games' / GAME_ID / 'extracted' / f'{logbook_doc_name}.json'

pages = json.loads(logbook_json.read_text())

augmented: list[tuple[int, int, str]] = []  # (page_num, bbox_index, mission_n)
for page in pages:
    page['doc_tag'] = logbook_tag  # restore tag so chunk_by_sections carries it through
    for i, bbox in enumerate(page.get('bboxes', [])):
        if bbox.get('label') != 'picture':
            continue
        m = PENTAGON_PATTERN.search(bbox.get('text', ''))
        if not m:
            continue
        n = m.group(1)
        label = f'Mission {n}'
        if bbox['text'].startswith(label):
            continue
        # Prepend so the label is the first thing the embedder sees.
        bbox['text'] = f"{label}. {bbox['text']}"
        augmented.append((page['page_num'], i, n))

print(f'Augmented {len(augmented)} pentagon bboxes:')
for page_num, bbox_idx, n in augmented:
    print(f'  page {page_num:>2d}  bbox[{bbox_idx:>2d}]  → Mission {n}')

Augmented 32 pentagon bboxes:
  page  5  bbox[ 1]  → Mission 4
  page  5  bbox[13]  → Mission 6
  page  6  bbox[ 1]  → Mission 7
  page  7  bbox[ 3]  → Mission 11
  page  7  bbox[15]  → Mission 12
  page  8  bbox[ 1]  → Mission 13
  page  8  bbox[10]  → Mission 14
  page  9  bbox[ 1]  → Mission 16
  page  9  bbox[ 9]  → Mission 17
  page  9  bbox[18]  → Mission 18
  page 10  bbox[ 1]  → Mission 19
  page 10  bbox[12]  → Mission 20
  page 10  bbox[22]  → Mission 21
  page 11  bbox[ 3]  → Mission 23
  page 11  bbox[12]  → Mission 24
  page 12  bbox[ 1]  → Mission 25
  page 12  bbox[14]  → Mission 26
  page 12  bbox[23]  → Mission 27
  page 13  bbox[14]  → Mission 29
  page 13  bbox[24]  → Mission 30
  page 14  bbox[ 1]  → Mission 31
  page 14  bbox[10]  → Mission 32
  page 14  bbox[19]  → Mission 33
  page 15  bbox[ 1]  → Mission 34
  page 15  bbox[12]  → Mission 35
  page 15  bbox[16]  → Mission 36
  page 17  bbox[ 1]  → Mission 40
  page 18  bbox[ 1]  → Mission 42
  page 18  bbox[10]  

In [5]:
# Re-chunk the augmented pages and replace the logbook's points in Qdrant.
# This re-embeds the chunks (since text changed), but doesn't touch the rulebook.
from boardgame_agent.rag.indexer import build_index, remove_doc_from_index
from boardgame_agent.rag.extractor import chunk_by_sections

new_chunks = chunk_by_sections(pages)
print(f'Re-chunked logbook into {len(new_chunks)} chunks')

remove_doc_from_index(logbook_doc_name, GAME_ID, client=client)
print(f'Removed existing logbook points from Qdrant')

build_index(new_chunks, client=client)
print(f'Indexed {len(new_chunks)} augmented logbook chunks')

Re-chunked logbook into 25 chunks
Removed existing logbook points from Qdrant
Indexed 25 augmented logbook chunks


In [6]:
# Re-run the failing query and check whether the labeled pages now appear in top-k.
labeled_pages = {pn for pn, _, _ in augmented}
print(f'Pages with at least one labeled pentagon: {sorted(labeled_pages)}')
print()

for query in [
    'how do you win mission 3',
    'Can you explain the criteria for winning on mission 3',
    'how do you win mission 12',
    'mission 25',
]:
    points = retrieve_pages(client, query, GAME_ID, k=5)
    print(f'─── {query!r}')
    for i, p in enumerate(points, 1):
        pl = p.payload
        is_logbook = 'Log_Book' in pl['doc_name']
        marker = ''
        if is_logbook and pl['page_num'] in labeled_pages:
            marker = '  ★ (has labeled pentagon)'
        text = pl.get('text', '')
        # Show whether the chunk text now contains 'Mission N'
        mission_in_text = re.findall(r'Mission \d+', text[:400])
        if mission_in_text:
            marker += f'  text contains: {sorted(set(mission_in_text))}'
        print(f'  {i}. {pl["doc_name"][:40]:40s} p.{pl["page_num"]:>2d}{marker}')
    print()

Pages with at least one labeled pentagon: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19]

─── 'how do you win mission 3'
  1. The_Crew_The_Quest_for_Planet_Nine.Rules p.21
  2. The_Crew_The_Quest_for_Planet_Nine.Rules p.16
  3. The_Crew_The_Quest_for_Planet_Nine.Rules p.15
  4. The_Crew_The_Quest_for_Planet_Nine.Rules p.10
  5. The_Crew_The_Quest_for_Planet_Nine.Rules p.12

─── 'Can you explain the criteria for winning on mission 3'
  1. The_Crew_The_Quest_for_Planet_Nine.Rules p.16
  2. The_Crew_The_Quest_for_Planet_Nine.Rules p.15
  3. The_Crew_The_Quest_for_Planet_Nine.Log_B p.14  ★ (has labeled pentagon)  text contains: ['Mission 31']
  4. The_Crew_The_Quest_for_Planet_Nine.Rules p.10
  5. The_Crew_The_Quest_for_Planet_Nine.Log_B p.15  ★ (has labeled pentagon)  text contains: ['Mission 34']

─── 'how do you win mission 12'
  1. The_Crew_The_Quest_for_Planet_Nine.Rules p.15
  2. The_Crew_The_Quest_for_Planet_Nine.Log_B p.12  ★ (has labeled pentagon)
  3. The_Crew_The_Quest_for_

In [9]:
# Side-by-side: also re-run the full expansion view on Mission 3 so you can
# see the bbox-level content of whatever page lands at the top.
show_retrieval_with_expansion(
    'Can you explain the criteria for winning on mission 3',
    doc_tag=None,
    k=5,
)

################################################################################
QUERY: 'Can you explain the criteria for winning on mission 3'    (doc_tag=None, k=5)
################################################################################

── #1  The_Crew_The_Quest_for_Planet_Nine.Rules p.16  (859 chars)  doc_tag=rulebook
   RETRIEVED CHUNK: 'The distribution of the task cards happens as previously described. If you take a task card, you must also take any corresponding task token.\n\nNumber and arrow tokens both specify an order. Arrow tokens give you more flexibility in fulfilli'
   (no siblings on this page — expansion would add nothing)

── #2  The_Crew_The_Quest_for_Planet_Nine.Rules p.15  (1416 chars)  doc_tag=rulebook
   RETRIEVED CHUNK: 'It can happen that several tasks with task tokens are won by a single crew member in the same trick. If these task tokens are consecutive, for example, and ,or and , both are considered to have been correctly fulfilled, 2\n\nregardless

### Interpreting

- **If a logbook page with a labeled pentagon appears in top-5 for the mission query** → mechanism is validated. Building out the full agentic pipeline (legend discovery + visual icon matching + confidence handling for all icon classes) is justified.
- **If labeled pages still don't appear** → the failure is upstream of labeling. Investigate the raw RRF candidates before rerank — if Mission-N pages don't even show up in the candidate pool, the embedding model isn't picking up the labels; if they show up and rerank drops them, the issue is rerank quality, not labeling.
- **If labeled pages appear but the answer still wouldn't be correct** → expected. This MVP only labels mission identifiers. The other icons on a mission page (red starburst = number of tasks, blue/black squares = task ordering) still carry only visual descriptions. Their definitions live elsewhere (logbook p.3 and rulebook p.14 respectively) and would need their own labeling pass.

### Revert

```python
from boardgame_agent.rag.indexer import reindex_all
reindex_all()
```

This rebuilds the entire Qdrant collection from cached Docling JSONs, restoring all original bbox descriptions. The JSON files were never modified — only the in-memory copy was augmented.